## Rubric

Instructions: DELETE this cell before you submit via a `git push` to your repo before deadline. This cell is for your reference only and is not needed in your report. 

Scoring: Out of 10 points

- Each Developing  => -2 pts
- Each Unsatisfactory/Missing => -4 pts
  - until the score is 

If students address the detailed feedback in a future checkpoint they will earn these points back


|                  | Unsatisfactory                                                                                                                                                                                                    | Developing                                                                                                                                                                                              | Proficient                                     | Excellent                                                                                                                              |
|------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------|
| Data relevance   | Did not have data relevant to their question. Or the datasets don't work together because there is no way to line them up against each other. If there are multiple datasets, most of them have this trouble | Data was only tangentially relevant to the question or a bad proxy for the question. If there are multiple datasets, some of them may be irrelevant or can't be easily combined.                       | All data sources are relevant to the question. | Multiple data sources for each aspect of the project. It's clear how the data supports the needs of the project.                         |
| Data description | Dataset or its cleaning procedures are not described. If there are multiple datasets, most have this trouble                                                                                              | Data was not fully described. If there are multiple datasets, some of them are not fully described                                                                                                      | Data was fully described                       | The details of the data descriptions and perhaps some very basic EDA also make it clear how the data supports the needs of the project. |
| Data wrangling   | Did not obtain data. They did not clean/tidy the data they obtained.  If there are multiple datasets, most have this trouble                                                                                 | Data was partially cleaned or tidied. Perhaps you struggled to verify that the data was clean because they did not present it well. If there are multiple datasets, some have this trouble | The data is cleaned and tidied.                | The data is spotless and they used tools to visualize the data cleanliness and you were convinced at first glance                      |


# COGS 108 - Data Checkpoint

## Authors

Instructions: REPLACE the contents of this cell with your team list and their contributions. Note that this will change over the course of the checkpoints

This is a modified [CRediT taxonomy of contributions](https://credit.niso.org). For each group member please list how they contributed to this project using these terms:
> Analysis, Background research, Conceptualization, Data curation, Experimental investigation, Methodology, Project administration, Software, Visualization, Writing – original draft, Writing – review & editing

Example team list and credits:
- Alice Anderson: Conceptualization, Data curation, Methodology, Writing - original draft
- Bob Barker:  Analysis, Software, Visualization
- Charlie Chang: Project administration, Software, Writing - review & editing
- Dani Delgado: Analysis, Background research, Visualization, Writing - original draft

## Research Question

Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your proposal feedback



## Background and Prior Work

Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your proposal feedback

## Hypothesis


Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your proposal feedback


## Data

### Data overview

For this checkpoint, we use one public-use survey dataset.

- Dataset Name: 2024 National Health Interview Survey (NHIS), Sample Adult public-use file
- Link to dataset: https://ftp.cdc.gov/pub/health_Statistics/nchs/Datasets/NHIS/2024/adult24csv.zip
- Documentation/codebook: https://ftp.cdc.gov/pub/health_statistics/nchs/dataset_documentation/NHIS/2024/Adult-codebook.pdf
- Number of observations: 32,629 sample adults, according to the CDC checksum/file list
- Number of variables in full raw file: checked in code after loading the CSV
- Unit of observation: one sampled U.S. adult respondent

The variables most relevant to this project measure transportation barriers to care, housing strain, housing tenure, government rent assistance, medical-care delays, usual source of care, emergency-room use, time since last doctor visit, self-rated health, age, sex, race/ethnicity, education, poverty ratio, health insurance coverage, and NHIS survey design variables. These variables let us study whether transportation and housing-related vulnerability are associated with disruptions in a patient's healthcare journey.

Important shortcomings: NHIS is cross-sectional, so this dataset can show associations but cannot prove that transportation or housing barriers caused worse healthcare access or health outcomes. Several variables are self-reported, which may introduce recall or reporting bias. The public-use data also suppresses or recodes some sensitive details, including geography, so this checkpoint cannot analyze neighborhood-level conditions directly. Finally, NHIS has complex survey weights; the checkpoint below keeps the survey-weight variables for later analysis but uses simple unweighted summaries only for basic wrangling checks.

In [1]:
# Setup code -- this only needs to be run once after cloning the repo.
# It downloads the NHIS 2024 Sample Adult CSV to data/00-raw/ if it is not already there.

from pathlib import Path
import urllib.request
import zipfile

raw_dir = Path('data/00-raw')
processed_dir = Path('data/02-processed')
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

nhis_zip_url = 'https://ftp.cdc.gov/pub/health_Statistics/nchs/Datasets/NHIS/2024/adult24csv.zip'
zip_path = raw_dir / 'adult24csv.zip'
raw_path = raw_dir / 'adult24.csv'

if not raw_path.exists():
    urllib.request.urlretrieve(nhis_zip_url, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extract('adult24.csv', path=raw_dir)

print(f'Raw NHIS file ready at: {raw_path}')

Raw NHIS file ready at: data/00-raw/adult24.csv


### 2024 NHIS Sample Adult Public-Use File

The National Health Interview Survey (NHIS) is an annual U.S. household survey run by the National Center for Health Statistics. We use the 2024 Sample Adult public-use file because it contains individual-level measures of healthcare access, transportation barriers, housing strain, insurance coverage, poverty, demographics, and health status. The raw file is already rectangular: each row is one sampled adult, and each column is a survey variable. For this project, we keep a smaller analysis table containing the variables most directly connected to our research question.

Important metrics include `TRANSPOR_A`, a binary-coded item indicating whether the respondent delayed care because they lacked reliable transportation; `HOUSECOST_A`, which indicates trouble paying housing costs; `HOUTENURE_A`, which distinguishes owned, rented, and other housing arrangements; and `HOUGVASST_A`, which indicates whether the government pays part of rent for eligible renters. Healthcare journey outcomes include delayed medical care in the past 12 months (`MEDDL12M_A`), lack of a usual place for care (`USUALPL_A`), emergency room visits in the past 12 months (`EMERG12MTC_A`), time since last doctor visit (`LASTDR_A`), and self-rated general health (`PHSTAT_A`). We also keep age, sex, race/ethnicity, education, poverty ratio, insurance coverage, and survey design variables as controls or future weighting variables.

The main concerns are that NHIS uses special numeric codes for nonresponse categories such as refused, not ascertained, and don't know, so those codes need to be converted to missing values before analysis. Some variables are only asked of certain respondents, especially housing-assistance questions that apply to renters, which creates structural missingness rather than random missingness. The survey is also cross-sectional and self-reported, so we should describe associations carefully and avoid causal claims.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [3]:
# Step 1: inspect the CSV header to see the raw NHIS variable names
all_columns = pd.read_csv(raw_path, nrows=0).columns.tolist()

print(f'Total columns in adult24.csv: {len(all_columns)}')
print('First 40 column names:')
print(all_columns[:40])

# Define candidate columns based on research question
candidate_columns = [
    'TRANSPOR_A',  # Delayed care due to no reliable transportation
    'HOUSECOST_A', # Housing cost burden
    'HOUTENURE_A', # Housing tenure
    'HOUGVASST_A', # Housing assistance
    'DELAYCOST_A', # Delayed care due to cost
    'USUALPL_A',   # Usual place for care
    'EMERG12MTC_A', # Emergency room visits in past 12 months
    'PHSTAT_A',    # Self-rated physical health status
    'AGEP_A',      # Age
    'SEX_A',       # Sex
    'RACEALLP_A'   # Race
]

print('\nCandidate columns present in the file:')
print([col for col in candidate_columns if col in all_columns])

Total columns in adult24.csv: 630
First 40 column names:
['RATCAT_A', 'INCTCFLG_A', 'IMPINCFLG_A', 'PPSU', 'PSTRAT', 'WLKLEISTC_A', 'WLKTRANTC_A', 'HISPALLP_A', 'RACEALLP_A', 'ANYDIFF_A', 'DISAB3_A', 'NUMBRN1TC_A', 'K6SPD_A', 'SCHDYMSSTC_A', 'AFNOW', 'REPWRKDYTC_A', 'YRSINUS_A', 'CITZNSTP_A', 'PRTNREDUCP_A', 'SPOUSEDUCP_A', 'LEGMSTAT_A', 'MARSTAT_A', 'SASPPRACE_A', 'SASPPHISP_A', 'PRTNRAGETC_A', 'SPOUSAGETC_A', 'PRTNRWKFT_A', 'PRTNRWRK_A', 'SPOUSWKFT_A', 'SPOUSWRK_A', 'SPOUSESEX_A', 'PRTNRSEX_A', 'INJWRKDYTC_A', 'NUMINJTC_A', 'SHINGYEARP_A', 'PROXYREL2_A', 'PROXYFLAG_A', 'HHRESPSA_FLG', 'PCNTADWFP1_A', 'PCNTADWKP1_A']

Candidate columns present in the file:
['TRANSPOR_A', 'HOUSECOST_A', 'HOUTENURE_A', 'HOUGVASST_A', 'USUALPL_A', 'EMERG12MTC_A', 'PHSTAT_A', 'AGEP_A', 'SEX_A', 'RACEALLP_A']


In [4]:
# Step 2: document what those selected variables mean using the official NHIS 2024 codebook
variable_dictionary = pd.DataFrame([
    ['HHX', 'Household ID', 'Identifier', 'Links each respondent to a household'],
    ['WTFA_A', 'Final annual sample-adult weight', 'Survey design', 'Needed for weighted NHIS analysis'],
    ['PSTRAT', 'Pseudo-stratum', 'Survey design', 'Used with weights for correct variance estimates'],
    ['PPSU', 'Pseudo-PSU', 'Survey design', 'Used with weights for correct variance estimates'],
    ['AGEP_A', 'Age of sample adult', 'Control', 'Basic demographic control'],
    ['SEX_A', 'Sex of sample adult', 'Control', 'Basic demographic control'],
    ['HISPALLP_A', 'Race/ethnicity recode', 'Control', 'Useful for disparity analysis'],
    ['RACEALLP_A', 'Race-group recode', 'Control', 'Useful for disparity analysis'],
    ['EDUCP_A', 'Educational level', 'Control', 'Socioeconomic control'],
    ['POVRATTC_A', 'Poverty ratio', 'Control', 'Income-related control'],
    ['HICOV_A', 'Health insurance coverage', 'Control', 'Access-to-care control'],
    ['TRANSPOR_A', 'Delayed care because no reliable transportation', 'Predictor', 'Direct transportation barrier measure'],
    ['HOUSECOST_A', 'Had trouble paying for housing', 'Predictor', 'Housing strain / home-environment proxy'],
    ['HOUTENURE_A', 'Residence owned or rented', 'Predictor', 'Housing stability / tenure measure'],
    ['HOUGVASST_A', 'Government paying part of rent', 'Predictor', 'Captures assisted rental context'],
    ['USUALPL_A', 'Has a usual place for care', 'Outcome', 'Tracks continuity of care'],
    ['MEDDL12M_A', 'Delayed medical care in past 12 months', 'Outcome', 'Direct patient-journey disruption measure'],
    ['EMERG12MTC_A', 'Number of ER visits in past 12 months', 'Outcome', 'Healthcare use / disruption signal'],
    ['LASTDR_A', 'Time since last saw doctor', 'Outcome', 'Tracks gaps in routine care'],
    ['PHSTAT_A', 'General health status', 'Outcome', 'High-level health outcome']
], columns=['Variable', 'Official meaning', 'Role in project', 'Why we kept it'])

variable_dictionary

,Variable,Official meaning,Role in project,Why we kept it
0,HHX,Household ID,Identifier,Links each respondent to a household
1,WTFA_A,Final annual sample-adult weight,Survey design,Needed for weighted NHIS analysis
2,PSTRAT,Pseudo-stratum,Survey design,Used with weights for correct variance estimates
3,PPSU,Pseudo-PSU,Survey design,Used with weights for correct variance estimates
4,AGEP_A,Age of sample adult,Control,Basic demographic control
5,SEX_A,Sex of sample adult,Control,Basic demographic control
6,HISPALLP_A,Race/ethnicity recode,Control,Useful for disparity analysis
7,RACEALLP_A,Race-group recode,Control,Useful for disparity analysis
8,EDUCP_A,Educational level,Control,Socioeconomic control
9,POVRATTC_A,Poverty ratio,Control,Income-related control


In [5]:
usecols = [
    'HHX', 'WTFA_A', 'PSTRAT', 'PPSU',
    'AGEP_A', 'SEX_A', 'HISPALLP_A', 'RACEALLP_A', 'EDUCP_A', 'POVRATTC_A', 'HICOV_A',
    'TRANSPOR_A', 'HOUSECOST_A', 'HOUTENURE_A', 'HOUGVASST_A',
    'USUALPL_A', 'MEDDL12M_A', 'EMERG12MTC_A', 'LASTDR_A', 'PHSTAT_A'
]

nhis = pd.read_csv(raw_path, usecols=usecols, low_memory=False)

print('NHIS loaded successfully')
print('Shape:', nhis.shape)
nhis.head()

NHIS loaded successfully
Shape: (32629, 20)


,PPSU,PSTRAT,HISPALLP_A,RACEALLP_A,EMERG12MTC_A,EDUCP_A,SEX_A,AGEP_A,TRANSPOR_A,HOUSECOST_A,HOUGVASST_A,HOUTENURE_A,MEDDL12M_A,USUALPL_A,LASTDR_A,HICOV_A,PHSTAT_A,WTFA_A,HHX,POVRATTC_A
0,2,122,2,1,0,5,1,49,2,2,NaN,1,2,2,5,2,1,5780.565,H067658,2.82
1,2,122,2,1,2,5,1,53,1,2,2.0,2,2,1,1,1,2,3994.244,H076577,2.01
2,2,122,2,1,0,5,1,82,2,2,2.0,2,2,1,1,1,2,6636.755,H019335,1.90
3,2,122,2,1,0,4,1,42,2,2,NaN,1,2,1,2,1,3,13767.420,H012701,4.48
4,1,115,3,2,1,9,2,38,2,2,2.0,2,2,1,1,1,3,18880.030,H049678,6.37


In [6]:
df = nhis.copy()

special_missing = {
    'SEX_A': [7, 9],
    'HISPALLP_A': [7],
    'RACEALLP_A': [7, 8, 9],
    'EDUCP_A': [97, 99],
    'HICOV_A': [7, 9],
    'TRANSPOR_A': [7, 8, 9],
    'HOUSECOST_A': [7, 8, 9],
    'HOUTENURE_A': [7, 8, 9],
    'HOUGVASST_A': [7, 8, 9],
    'USUALPL_A': [7, 8, 9],
    'MEDDL12M_A': [7, 8, 9],
    'EMERG12MTC_A': [7, 8, 9],
    'LASTDR_A': [7, 8, 9],
    'PHSTAT_A': [7, 8, 9],
}

for col, values in special_missing.items():
    df[col] = df[col].replace(values, np.nan)

sex_map = {1: 'Male', 2: 'Female'}
race_ethnicity_map = {
    1: 'Hispanic',
    2: 'White non-Hispanic',
    3: 'Black non-Hispanic',
    4: 'Asian non-Hispanic',
    5: 'Other non-Hispanic',
    6: 'Multiple race non-Hispanic'
}
race_map = {
    1: 'White only',
    2: 'Black only',
    3: 'Asian only',
    4: 'AIAN only',
    5: 'Other race only',
    6: 'Multiple race'
}
education_map = {
    1: 'Less than HS',
    2: 'GED',
    3: 'HS graduate',
    4: 'Some college, no degree',
    5: 'AA academic',
    6: 'AA technical',
    7: "Bachelor's",
    8: "Master's",
    9: 'Professional',
    10: 'Doctorate'
}
health_map = {
    1: 'Excellent',
    2: 'Very good',
    3: 'Good',
    4: 'Fair',
    5: 'Poor'
}
tenure_map = {
    1: 'Owned or being bought',
    2: 'Rented',
    3: 'Other arrangement'
}

df['sex'] = df['SEX_A'].map(sex_map)
df['race_ethnicity'] = df['HISPALLP_A'].map(race_ethnicity_map)
df['race_group'] = df['RACEALLP_A'].map(race_map)
df['education'] = df['EDUCP_A'].map(education_map)
df['general_health'] = df['PHSTAT_A'].map(health_map)
df['housing_tenure'] = df['HOUTENURE_A'].map(tenure_map)

missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

missing_summary.head(15)


,missing_count,missing_pct
HOUGVASST_A,23230,71.19
RACEALLP_A,1700,5.21
race_group,1700,5.21
HOUSECOST_A,1455,4.46
TRANSPOR_A,1407,4.31
housing_tenure,1384,4.24
HOUTENURE_A,1384,4.24
HISPALLP_A,411,1.26
race_ethnicity,411,1.26
EMERG12MTC_A,280,0.86


Cleaning choices: the NHIS codebook uses numeric values such as 7, 8, 9, 97, and 99 for refused, not ascertained, and don't know responses. These are not meaningful measured values, so we convert them to `NaN` before creating analysis variables. We do not drop all rows with missing values at this checkpoint because some missingness is structural. For example, `HOUGVASST_A` is mainly relevant for renters, so homeowners are later coded as 0 for government rent assistance while true nonresponse remains missing. For summary indexes, we use `sum(..., min_count=1)` so respondents with at least one valid component can still contribute, while rows with no valid component remain missing.

In [7]:
# Quick look at the cleaned core variables
core_vars = [
    'TRANSPOR_A', 'HOUSECOST_A', 'HOUTENURE_A', 'HOUGVASST_A',
    'USUALPL_A', 'MEDDL12M_A', 'EMERG12MTC_A', 'LASTDR_A', 'PHSTAT_A',
    'POVRATTC_A', 'HICOV_A'
]

print('Raw value counts after converting special nonresponse codes to NaN:')
for col in core_vars:
    print(f'\n{col}')
    print(df[col].value_counts(dropna=False).sort_index())

range_checks = pd.DataFrame({
    'age_below_18': [int((df['AGEP_A'] < 18).sum())],
    'age_above_99': [int((df['AGEP_A'] > 99).sum())],
    'negative_poverty_ratio': [int((df['POVRATTC_A'] < 0).sum())],
    'er_visits_above_10': [int((df['EMERG12MTC_A'] > 10).sum())]
})

range_checks

Raw value counts after converting special nonresponse codes to NaN:

TRANSPOR_A
TRANSPOR_A
1.0     1899
2.0    29323
NaN     1407
Name: count, dtype: int64

HOUSECOST_A
HOUSECOST_A
1.0     2195
2.0    28979
NaN     1455
Name: count, dtype: int64

HOUTENURE_A
HOUTENURE_A
1.0    21183
2.0     9450
3.0      612
NaN     1384
Name: count, dtype: int64

HOUGVASST_A
HOUGVASST_A
1.0     1337
2.0     8062
NaN    23230
Name: count, dtype: int64

USUALPL_A
USUALPL_A
1.0    28930
2.0     2739
3.0      721
NaN      239
Name: count, dtype: int64

MEDDL12M_A
MEDDL12M_A
1.0     2564
2.0    29791
NaN      274
Name: count, dtype: int64

EMERG12MTC_A
EMERG12MTC_A
0.0    25399
1.0     4545
2.0     1474
3.0      458
4.0      473
NaN      280
Name: count, dtype: int64

LASTDR_A
LASTDR_A
0.0       53
1.0    28187
2.0     2089
3.0      713
4.0      600
5.0      411
6.0      320
NaN      256
Name: count, dtype: int64

PHSTAT_A
PHSTAT_A
1.0     6302
2.0    10919
3.0    10037
4.0     4147
5.0     1208
NaN       

,age_below_18,age_above_99,negative_poverty_ratio,er_visits_above_10
0,0,0,0,0


In [8]:
df['transportation_barrier'] = np.where(df['TRANSPOR_A'].notna(), (df['TRANSPOR_A'] == 1).astype(int), np.nan)
df['housing_cost_trouble'] = np.where(df['HOUSECOST_A'].notna(), (df['HOUSECOST_A'] == 1).astype(int), np.nan)
df['renter_or_other'] = np.where(df['HOUTENURE_A'].notna(), df['HOUTENURE_A'].isin([2, 3]).astype(int), np.nan)

# HOUGVASST_A is only asked of renters. We keep homeowners / non-renters as 0 here.
df['government_rent_assistance'] = np.where(df['HOUGVASST_A'] == 1, 1,
                                    np.where(df['HOUGVASST_A'] == 2, 0, np.nan))
df.loc[df['HOUTENURE_A'].isin([1, 3]), 'government_rent_assistance'] = 0

df['low_income'] = np.where(df['POVRATTC_A'].notna(), (df['POVRATTC_A'] < 2.0).astype(int), np.nan)
df['housing_vulnerability_index'] = df[
    ['housing_cost_trouble', 'renter_or_other', 'government_rent_assistance', 'low_income']
].sum(axis=1, min_count=1)

df['delayed_medical_care'] = np.where(df['MEDDL12M_A'].notna(), (df['MEDDL12M_A'] == 1).astype(int), np.nan)
df['no_usual_place_of_care'] = np.where(df['USUALPL_A'].notna(), df['USUALPL_A'].isin([2, 3]).astype(int), np.nan)
df['any_er_visit'] = np.where(df['EMERG12MTC_A'].notna(), (df['EMERG12MTC_A'] >= 1).astype(int), np.nan)
df['doctor_gap_over_1yr'] = np.where(df['LASTDR_A'].notna(), df['LASTDR_A'].isin([4, 5, 6]).astype(int), np.nan)
df['poor_or_fair_health'] = np.where(df['PHSTAT_A'].notna(), df['PHSTAT_A'].isin([4, 5]).astype(int), np.nan)
df['uninsured'] = np.where(df['HICOV_A'].notna(), (df['HICOV_A'] == 2).astype(int), np.nan)

df['patient_journey_disruption_index'] = df[
    ['delayed_medical_care', 'no_usual_place_of_care', 'any_er_visit', 'doctor_gap_over_1yr']
].sum(axis=1, min_count=1)

analysis_vars = [
    'transportation_barrier', 'housing_cost_trouble', 'renter_or_other',
    'government_rent_assistance', 'low_income', 'housing_vulnerability_index',
    'delayed_medical_care', 'no_usual_place_of_care', 'any_er_visit',
    'doctor_gap_over_1yr', 'poor_or_fair_health', 'uninsured',
    'patient_journey_disruption_index'
]

keep_cols = [
    'HHX', 'WTFA_A', 'PSTRAT', 'PPSU', 'AGEP_A', 'sex', 'race_ethnicity', 'race_group',
    'education', 'POVRATTC_A', 'uninsured', 'housing_tenure', 'general_health'
] + analysis_vars

clean_nhis = df[keep_cols].copy()
clean_nhis.to_csv(processed_dir / 'nhis_2024_sample_adult_cleaned.csv', index=False)

print(f'Cleaned analysis file saved to: {processed_dir / "nhis_2024_sample_adult_cleaned.csv"}')
print('Cleaned shape:', clean_nhis.shape)

df[analysis_vars].describe().T

Cleaned analysis file saved to: data/02-processed/nhis_2024_sample_adult_cleaned.csv
Cleaned shape: (32629, 26)


,count,mean,std,min,25%,50%,75%,max
transportation_barrier,31222.0,0.060822,0.239008,0.0,0.0,0.0,0.0,1.0
housing_cost_trouble,31174.0,0.070411,0.255843,0.0,0.0,0.0,0.0,1.0
renter_or_other,31245.0,0.322036,0.467264,0.0,0.0,0.0,1.0,1.0
government_rent_assistance,31194.0,0.042861,0.202546,0.0,0.0,0.0,0.0,1.0
low_income,32629.0,0.282387,0.450167,0.0,0.0,0.0,1.0,1.0
housing_vulnerability_index,32629.0,0.699010,0.911510,0.0,0.0,0.0,1.0,4.0
delayed_medical_care,32355.0,0.079246,0.270126,0.0,0.0,0.0,0.0,1.0
no_usual_place_of_care,32390.0,0.106823,0.308893,0.0,0.0,0.0,0.0,1.0
any_er_visit,32349.0,0.214844,0.410721,0.0,0.0,0.0,0.0,1.0
doctor_gap_over_1yr,32373.0,0.041115,0.198558,0.0,0.0,0.0,0.0,1.0


In [9]:
variables_summary = pd.DataFrame([
    ['Predictor', 'transportation_barrier', 'Binary', '1 = delayed care because no reliable transportation'],
    ['Predictor', 'housing_cost_trouble', 'Binary', '1 = had trouble paying for housing'],
    ['Predictor', 'renter_or_other', 'Binary', '1 = not owner-occupied housing'],
    ['Predictor', 'government_rent_assistance', 'Binary', '1 = government pays part of rent'],
    ['Predictor', 'low_income', 'Binary', '1 = poverty ratio below 2.0'],
    ['Predictor', 'housing_vulnerability_index', 'Count 0-4', 'Composite of housing strain indicators'],
    ['Outcome', 'delayed_medical_care', 'Binary', '1 = delayed medical care in past 12 months'],
    ['Outcome', 'no_usual_place_of_care', 'Binary', '1 = lacks usual place for care'],
    ['Outcome', 'any_er_visit', 'Binary', '1 = one or more ER visits in past 12 months'],
    ['Outcome', 'doctor_gap_over_1yr', 'Binary', '1 = last saw doctor more than 1 year ago'],
    ['Outcome', 'poor_or_fair_health', 'Binary', '1 = self-rated health is fair or poor'],
    ['Outcome', 'patient_journey_disruption_index', 'Count 0-4', 'Composite disruption score'],
    ['Control', 'AGEP_A', 'Continuous', 'Age in years'],
    ['Control', 'SEX_A', 'Categorical', 'Sex'],
    ['Control', 'HISPALLP_A', 'Categorical', 'Race/ethnicity recode'],
    ['Control', 'EDUCP_A', 'Ordinal', 'Education level'],
    ['Control', 'POVRATTC_A', 'Continuous', 'Poverty ratio'],
    ['Control', 'HICOV_A', 'Binary', 'Insurance coverage status']
], columns=['Category', 'Variable', 'Type', 'Why it matters'])

variables_summary

,Category,Variable,Type,Why it matters
0,Predictor,transportation_barrier,Binary,1 = delayed care because no reliable transport...
1,Predictor,housing_cost_trouble,Binary,1 = had trouble paying for housing
2,Predictor,renter_or_other,Binary,1 = not owner-occupied housing
3,Predictor,government_rent_assistance,Binary,1 = government pays part of rent
4,Predictor,low_income,Binary,1 = poverty ratio below 2.0
5,Predictor,housing_vulnerability_index,Count 0-4,Composite of housing strain indicators
6,Outcome,delayed_medical_care,Binary,1 = delayed medical care in past 12 months
7,Outcome,no_usual_place_of_care,Binary,1 = lacks usual place for care
8,Outcome,any_er_visit,Binary,1 = one or more ER visits in past 12 months
9,Outcome,doctor_gap_over_1yr,Binary,1 = last saw doctor more than 1 year ago


In [10]:
# Basic prevalence estimates from the cleaned 2024 NHIS sample adult file
prevalence = (df[analysis_vars].mean() * 100).round(1).sort_values(ascending=False)
prevalence


housing_vulnerability_index         69.9
patient_journey_disruption_index    44.1
renter_or_other                     32.2
low_income                          28.2
any_er_visit                        21.5
poor_or_fair_health                 16.4
no_usual_place_of_care              10.7
delayed_medical_care                 7.9
housing_cost_trouble                 7.0
uninsured                            6.3
transportation_barrier               6.1
government_rent_assistance           4.3
doctor_gap_over_1yr                  4.1
dtype: float64

In [11]:
# Simple unweighted comparisons to show the pattern in the data
transport_compare = df.groupby('transportation_barrier')[
    ['delayed_medical_care', 'no_usual_place_of_care', 'any_er_visit', 'poor_or_fair_health']
].mean().round(3)

housing_compare = df.groupby('housing_cost_trouble')[
    ['delayed_medical_care', 'no_usual_place_of_care', 'any_er_visit', 'poor_or_fair_health']
].mean().round(3)

print('Outcomes by transportation barrier')
print(transport_compare)
print('\nOutcomes by housing-cost trouble')
print(housing_compare)

Outcomes by transportation barrier
                        delayed_medical_care  no_usual_place_of_care  \
transportation_barrier                                                 
0.0                                    0.071                   0.103   
1.0                                    0.203                   0.144   

                        any_er_visit  poor_or_fair_health  
transportation_barrier                                     
0.0                            0.204                0.151  
1.0                            0.353                0.354  

Outcomes by housing-cost trouble
                      delayed_medical_care  no_usual_place_of_care  \
housing_cost_trouble                                                 
0.0                                  0.065                   0.103   
1.0                                  0.268                   0.148   

                      any_er_visit  poor_or_fair_health  
housing_cost_trouble                                     
0.0  

### Additional datasets

We are not using a second dataset for this checkpoint. The 2024 NHIS Sample Adult file already contains the transportation, housing, healthcare-access, health-status, and demographic variables needed for the first stage of the project, so no dataset join is required yet.

## Ethics

Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your proposal feedback

## Team Expectations 

Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your proposal feedback


## Project Timeline Proposal

Instructions: Replace this with your timeline.  **PLEASE UPDATE your Timeline!** No battle plan survives contact with the enemy, so make sure we understand how your plans have changed.  Also if you have lost points on the previous checkpoint fix them